# Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

  
if Path.cwd().name in ['ETL', 'data_science']:
    root = Path.cwd().parent
else:
    root = Path.cwd()

if str(root) not in sys.path:
    sys.path.append(str(root))
else:
    None
from utils import align_working_dir, save_chart

align_working_dir('data_science') 

%run ./0_content_load.ipynb

In [ ]:
# Save carts to a specific folder with a given prefix n the name.
CHART_PREFIX = "blog_post_"

#by Categories Annual Thematic Evolution 

In [ ]:
blog_mix_data = combined_df[combined_df['Type'] == 'Blog Post'].copy()

# Create Pivot Table and Normalize to 100%
# We use 'Year' and 'Category' from combined_df
pivot_df = blog_mix_data.groupby(['Year', 'Category']).size().unstack(fill_value=0)
pivot_pct = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

category_totals = pivot_df.sum()  # Total count per category
total_blog_posts = int(category_totals.sum())  # Grand total of all blog_posts

# --- Plotting ---
ax = pivot_pct.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='tab10',
width=0.8, rot=0)

fig = ax.get_figure()

fig.suptitle('Blog Posts: Annual Thematic Evolution')
ax.set_title(f'Based on the primary category\n Total Blog Posts: {total_blog_posts}', linespacing=1.5)

ax.set_ylabel('Percentage of Total Posts (%)')
ax.set_xlabel('Year')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Prepare the legend labels with counts
# This creates a list like: ['Category A (n=150)', 'Category B (n=200)', ...]
legend_labels = [f'{col} (n={int(category_totals[col])})' for col in pivot_pct.columns]

# Apply the custom legend labels
ax.legend(
    legend_labels, 
    title='Blog Post Categories:', 
    bbox_to_anchor=(1.02, 1), 
    loc='upper left',
    frameon=False,
    alignment='left'
)

# Add percentage labels inside the bars
for p in ax.patches:
    h = p.get_height()
    if h > 5: # Only label if there's enough space
        ax.annotate(f'{h:.0f}%',
                    (p.get_x() + p.get_width()/2., p.get_y() + h/2.),
                    ha='center', va='center', fontsize=16, color='white', fontweight='medium')

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "annual_thematic_evolution", CHART_PREFIX)
plt.show()

# Publishing Routine (Day of the Week)
Does the organization have a specific day they prefer to publish blog posts? This reveals their operational habit.

In [ ]:
blog_posts_anal_df = combined_df[combined_df['Type'] == 'Blog Post'].copy()

# Extract Day of Week (Same as before)
blog_posts_anal_df['Day_of_Week'] = blog_posts_anal_df['Date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

sns.countplot(
    data=blog_posts_anal_df,
    x='Day_of_Week',
    order=day_order,
    color=CUSTOM_PALETTE["Blog Post"],   # Assign hue to avoid warning
)

fig.suptitle('Publishing Routine')
ax.set_title('Which day are Blog Posts released during?\n Period: 2019-2025', linespacing=1.5)

ax.set_ylabel('Number of Blog Posts')
ax.set_xlabel('Days of the Week')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "publishing_routine", CHART_PREFIX)
plt.show()

# Top 10 Most Frequent Blog Post Tags

In [ ]:
# "Explode" the tags list so each tag gets its own row
# Add to a new cell
all_tags = blog_posts_df.explode('tags')
top_tags = all_tags['tags'].value_counts().head(10).reset_index()
top_tags.columns = ['Tag', 'Count']

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
       data=top_tags,
       y='Tag',
       x='Count',
    color=CUSTOM_PALETTE["Blog Post"]  
   )

sns.barplot(data=top_tags, y='Tag', x='Count', color=CUSTOM_PALETTE["Blog Post"]  )

fig.suptitle('Top 10 Most Frequent Blog Post Tags')
ax.set_title("Period: 2019-2025")

plt.xlabel('Occurrences')
plt.ylabel('')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "top_tags", CHART_PREFIX)

plt.show()

# Annual Category Distribution paired with Views and Likes

In [ ]:
# Aggregate for performance overview
blog_summary = (
    blog_posts_df.groupby('Primary_Cat')
    .agg(Count=('title', 'count'), Views=('views', 'sum'), Likes=('likes', 'sum'))
    .sort_values('Count', ascending=False)
    .reset_index()
)
# Setup Subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 9), sharey=True,
                                     gridspec_kw={'width_ratios': [2, 1.2, 1.2]})
# --- Chart 1: Post Count ---
sns.barplot(data=blog_summary, y='Primary_Cat', x='Count', color=CUSTOM_PALETTE['Blog Post'], ax=ax1)
ax1.set_title("Number of Posts")

# --- Chart 2: Total Views ---
sns.barplot(data=blog_summary, y='Primary_Cat', x='Views', color='#ff3a6f', ax=ax2)
ax2.set_title("Total Views")

# --- Chart 3: Total Likes ---
sns.barplot(data=blog_summary, y='Primary_Cat', x='Likes', color='#ffca3a', ax=ax3)
ax3.set_title("Total Likes")

# Add data labels
for ax in [ax1, ax2, ax3]:
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.tick_params(axis='both', which='major', left=True, bottom=True)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_width())}', (p.get_width(), p.get_y() +
p.get_height() / 2.),
                    ha='left', va='center', fontsize=14, xytext=(5, 0),
textcoords='offset points')
        
fig.suptitle('Blog Performance Overview by Primary Category', x=0.1, y=0.98, ha='left')
fig.text(0.1, 0.90, 'Which Category Drive the Most Engagement?\nTotal aggregated performance | Period: 2019-2025', ha='left', fontsize=14, linespacing=1.5, alpha=0.7)

plt.tight_layout(rect=[0, 0, 1, 0.93]) # Automated spacing that respects titles

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "performance_by_primary_category", CHART_PREFIX)
plt.show()

# Engagement Rate (The "Loyalty" Metric):
   * **Logic**: Calculate Likes / Views.
   * **Insight**: A post might get 10,000 views because of a catchy title, but only 5 likes. Another might get 100
     **views** but 50 likes. The latter is your "High-Loyalty" content.
   * **Goal**: Identify which topics actually resonate emotionally with the reader.

In [ ]:
# Aggregate using the standardized 'Primary_Cat'
blog_cat_stats = blog_posts_df.groupby('Primary_Cat').agg(
    Total_Views=('views', 'sum'),
    Total_Likes=('likes', 'sum')
).reset_index()

# Calculate Rate
blog_cat_stats['Engagement_Rate'] = (
    (blog_cat_stats['Total_Likes'] / blog_cat_stats['Total_Views'])
    .fillna(0) * 100
)

# 5. Sort for visualization
blog_cat_stats = blog_cat_stats.sort_values('Engagement_Rate', ascending=False)

# --- Plotting ---
fig, ax = plt.subplots(figsize=(12, 7))

sns.barplot(
    data=blog_cat_stats,
    x='Engagement_Rate',
    y='Primary_Cat',
    color=CUSTOM_PALETTE['Blog Post'],
    legend=False
)
# Add precision labels
for p in plt.gca().patches:
    rate = p.get_width()
    if rate > 0:
        plt.gca().annotate(f'{rate:.2f}%',
                           (rate, p.get_y() + p.get_height() / 2.),
                           ha='left', va='center', fontsize=10, xytext=(5, 0),
                           textcoords='offset points')

fig.suptitle('Engagement Rate by Lead Category')
ax.set_title('Likes as a percentage of Total Views')

ax.set_xlabel('Engagement Rate (%)')
ax.set_ylabel('Primary Category')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "engagement_rate", CHART_PREFIX)
plt.show()

# Editorial Depth (Distribution)
This identifies the "Standard length" of a blog post.

In [ ]:
blog_posts_word_count_df = blog_posts_df.query("publishedDate.dt.year != 2026").copy()

# Feature Engineering: Calculate Word Count
blog_posts_word_count_df['Word_Count'] = blog_posts_word_count_df['contentText'].fillna('').apply(lambda
x: len(str(x).split()))

# Statistical analysis
mean_val = blog_posts_word_count_df['Word_Count'].mean()
median_val = blog_posts_word_count_df['Word_Count'].median()

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(blog_posts_word_count_df['Word_Count'], bins=15, kde=True,
color=CUSTOM_PALETTE["Blog Post"])

# Add statistical markers
ax.axvline(mean_val, color='red', linestyle='--', label=f'Mean (Avg): {mean_val:.0f}')
ax.axvline(median_val, color='green', linestyle='-', label=f'Median (Midle):{median_val:.0f}')

fig.suptitle('Blog Post Length Distribution')
ax.set_title('Mean vs Median Depth Analysis\n Period: 2019-2025', linespacing=1.5)

ax.set_xlabel('Estimated Word Count')
ax.set_ylabel('Number of Posts')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

ax.legend()

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "length", CHART_PREFIX)
plt.show()

# Annual Blog Post Activity Pulse

In [ ]:
# Filter for events and create the pivot table
blog_post_pulse_df = (
       combined_df.query("Type == 'Blog Post'")
       .groupby(['Year', 'Month'])
       .size()
       .unstack(fill_value=0)
)

# Ensure all 12 months are represented even if no events happened
for month in range(1, 13):
       if month not in blog_post_pulse_df.columns:
           blog_post_pulse_df[month] = 0

# REVERSE the order of the years (rows) 
# This puts the most recent year (2025) at the top
blog_post_pulse_df = blog_post_pulse_df.iloc[::-1]

# Sort months correctly (1-12)
blog_post_pulse_df = blog_post_pulse_df.reindex(columns=range(1, 13))

# 3. Map month numbers to names for the x-axis
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep',
'Oct', 'Nov', 'Dec']
blog_post_pulse_df.columns = month_names

# --- Plotting ---
fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(
       blog_post_pulse_df,
       annot=True,
       fmt='d',
       cmap='PuBu',
       linewidths=.5,
       cbar_kws={'label': 'Number of Blog Posts Published'},
       ax=ax,
       annot_kws={"size": 12, "weight": "bold"}
)

# Manually turn on ONLY the left and bottom spines
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)

for spine in ['left', 'bottom']:
    ax.spines[spine].set_color('#292929') 
    ax.spines[spine].set_linewidth(1.0)

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

fig.suptitle('Annual Blog Post Activity Pulse', x=0.08, y=0.98, ha='left')
ax.set_title(f'Frequency of blog posts published per month across the years | Total: {total_blog_posts}', loc='left', pad=20)

ax.set_ylabel('Year')
ax.set_xlabel('Month of Year')

# Ensure the Y-axis years are horizontal and readable
plt.yticks(rotation=0)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "annual_activity_pulse", CHART_PREFIX)
plt.show()